# Aktywacja Strategii Odbicie Ogórkowe
Ten notatnik służy do testowania, wizualizacji i optymalizacji strategii powrotu do średniej po mocnych spadkach.

## Importy, dane i sygnały

In [1]:
import os
import sys

# Dodajemy folder glowny do path aby moduly dzialaly
sys.path.append(os.path.abspath('d:/Antigravity/rebound/lyse-lby'))
os.chdir('d:/Antigravity/rebound/lyse-lby')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm

# Importujemy ladowanie danych i nasze nowe moduly
from maciex_py.ladowanie_danych import create_stock_dfs
from odbicie.mackowe_sygnaly import mackowe_sygnaly
from odbicie.odbicie import generate_odbicie_entries
from odbicie.tbm import moving_triple_barrier_labels

# Ustawienia ładowania danych
settings = {
    'market': 'stocks',
    'interval': '1week',
    'vol_enabled': True,
    'vol_ratio_window': 20,
    'vol_ratio_threshold': 1.2,
    'cmo_enabled': True,
    'cmo_len': 6,
    'cmo_thres': -35,
    'cmo_thres_prev': -50
}

In [2]:
# 1. Ładowanie Danych
import pickle
import os

data_cache_file = 'dfs_cache.pkl'

if os.path.exists(data_cache_file):
    print("Znaleziono zapisane dane. Wczytywanie z pliku...")
    with open(data_cache_file, 'rb') as f:
        dfs_1d, dfs_1w = pickle.load(f)
    print(f"Wczytano {len(dfs_1d)} symboli 1D i {len(dfs_1w)} symboli 1W z pliku {data_cache_file}.")
else:
    print("Ładowanie danych dziennych i tygodniowych...")
    dfs_1d, dfs_1w = create_stock_dfs(settings)
    print(f"Załadowano {len(dfs_1d)} symboli 1D i {len(dfs_1w)} symboli 1W.")
    print("Zapisywanie danych do pliku...")
    with open(data_cache_file, 'wb') as f:
        pickle.dump((dfs_1d, dfs_1w), f)
    print("Dane zapisane pomyślnie.")


In [3]:
# 2. Generowanie Sygnałów Bazowych (mackowe_sygnaly)
signals_cache_file = 'signals_cache.pkl'

if os.path.exists(signals_cache_file):
    print("Znaleziono zapisane sygnały. Wczytywanie z pliku...")
    with open(signals_cache_file, 'rb') as f:
        signals_df = pickle.load(f)
    print(f"Wczytano {len(signals_df)} sygnałów z pliku {signals_cache_file}.")
else:
    signals_df = mackowe_sygnaly(
        dfs=dfs_1w,
        settings=settings,
        require_vol_confirmation=True,
        require_cmo_confirmation=True,
        interval='1w',
        entry_offset=0,
        pattern_cols=['hammer', 'inverted_hammer', 'engulfing_bull', 'piercing_line'],
        debug=True
    )
    print("Zapisywanie sygnałów do pliku...")
    with open(signals_cache_file, 'wb') as f:
        pickle.dump(signals_df, f)
    print("Sygnały zapisane pomyślnie.")

signals_df.describe()


## Wejście i Wyjście

In [20]:
# 3. Wejście na podstawie progu (Odbicie Ogórkowe)
threshold_pct = 0.11  # 3% spadek od zamknięcia świecy sygnałowej

entries_df = generate_odbicie_entries(
    signals_df=signals_df,
    market_data_daily=dfs_1d,
    threshold_pct=threshold_pct,
    max_setup_hold_bars=10
)
print(f"Wygenerowano {len(entries_df)} wejść przy progu {threshold_pct*100}%")
entries_df.head()

Wygenerowano 107 wejść przy progu 11.0%


,symbol,signal_time,pattern,entry_time,entry_price,signal_close,threshold_pct,entry_atr
0,ABNB,2022-06-19,hammer,2022-06-30,88.546098,99.489998,0.11,6.897889
1,ALB,2023-11-05,hammer,2023-11-09,113.902203,127.980003,0.11,7.207769
2,ALB,2023-10-29,inverted_hammer,2023-11-01,119.651602,134.440002,0.11,7.000488
3,ARE,2025-11-23,hammer,2025-12-08,45.292099,50.889999,0.11,2.208598
4,AXP,2025-03-23,engulfing_bull,2025-04-04,237.919998,270.510010,0.11,10.113220


In [136]:
# Params           Więcej:                                                              Mniej:
tpm = 1.25    #    - łapiemy większe ruchy (ryzykujemy powrotem).                        - ratujemy i szybciej zbieramy mniejsze kwoty.
slm = 2.5     #    - luźniejszy stop loss (wytrzymuje szum korekcyjny).                  - szybsza kapitulacja i ucinanie straty z palca.
ttpm = 0.25   #    - luźniejsze spuszczanie kursu w trendach, nie zostajemy wyrzuceni.   - szybsze zabezpieczanie małego peaku.
mhb = 15      #    - dajemy kapitałowi długo leżeć pod ruchem bocznym.                   - szukamy szybkich obrotów uwalnaijąc portfel.

In [ ]:
# Params II
tpm = 2.6
slm = 3.6
ttpm = 0.1
mhb = 22


In [152]:
# Params III
tpm = 1.1
slm = 3.2
ttpm = 0.1  
mhb = 15

In [159]:
# Params IV => overfitted w ciul
tpm = 1.13
slm = 3.14
ttpm = 0.1
mhb = 15

In [180]:
# Params V => overfitted, ale z głową
tpm = 1.2
slm = 2.7
ttpm = 0.1
mhb = 15

In [181]:
# 4. Wyjście z użyciem Moving Triple Barrier Method
trades_df = moving_triple_barrier_labels(
    entries_df=entries_df,
    market_data_daily=dfs_1d,
    tp_mult=tpm,
    sl_mult=slm,
    tp_trail_mult=ttpm,
    max_holding_bars=mhb
)
print(f"Zakończono {len(trades_df)} transakcji.")
trades_df.head()

Zakończono 107 transakcji.


,symbol,signal_time,pattern,entry_time,entry_price,signal_close,threshold_pct,entry_atr,exit_time,exit_price,return_pct,exit_reason,hold_bars
0,ABNB,2022-06-19,hammer,2022-06-30,88.546098,99.489998,0.11,6.897889,2022-07-08,96.860214,9.389591,TRAILING_TP,5
1,ALB,2023-11-05,hammer,2023-11-09,113.902203,127.980003,0.11,7.207769,2023-11-16,133.739230,17.415841,TRAILING_TP,5
2,ALB,2023-10-29,inverted_hammer,2023-11-01,119.651602,134.440002,0.11,7.000488,2023-11-06,133.519952,11.590610,TRAILING_TP,3
3,ARE,2025-11-23,hammer,2025-12-08,45.292099,50.889999,0.11,2.208598,2025-12-19,48.949138,8.074342,TRAILING_TP,9
4,AXP,2025-03-23,engulfing_bull,2025-04-04,237.919998,270.510010,0.11,10.113220,2025-04-10,263.758667,10.860234,TRAILING_TP,4


## Analiza

In [182]:
# 5. Analiza i Statystyki
if not trades_df.empty:
    wins = (trades_df['return_pct'] > 0).sum()
    losses = (trades_df['return_pct'] <= 0).sum()
    win_rate = wins / len(trades_df) * 100
    
    print(f"Total Trades: {len(trades_df)}")
    print(f"Win Rate: {win_rate:.2f}%")
    print(f"Avg Return: {trades_df['return_pct'].mean():.2f}%")
    print(f"Avg bars held: {trades_df['hold_bars'].mean():.2f}")

    
    # Powody wyjścia
    print("\nExit Reasons:")
    print(trades_df['exit_reason'].value_counts())
else:
    print("Brak transakcji do analizy.")

Total Trades: 107
Win Rate: 78.50%
Avg Return: 5.19%
Avg bars held: 6.79

Exit Reasons:
exit_reason
TRAILING_TP    80
TIME_EXIT      13
TRAILING_SL    11
SL              3
Name: count, dtype: int64


## Ploty

In [ ]:
module_path = r"d:\Antigravity\rebound\lyse-lby\odbicie"
if module_path not in sys.path:
    sys.path.append(module_path)
    
from candlestick_cell import show_trade_viewer
show_trade_viewer(trades_df, dfs_1d, tpm, slm, ttpm, mhb)

Output()

## Optymalizacja

In [111]:
# 6. Optymalizacja Progu Wejścia i Czasu Trzymania Setupu
import itertools
from tqdm.notebook import tqdm

def optimize_threshold(thresholds, max_holding_bars):
    results = []
    
    # Tworzymy siatkę wszystkich kombinacji wejściowych list
    grid = list(itertools.product(thresholds, max_holding_bars))
    
    for th, max_bars in tqdm(grid, desc="Optymalizacja progu"):
        # max_bars definiuje ile dni po sygnale czekamy na wpadnięcie w próg
        ents = generate_odbicie_entries(signals_df, dfs_1d, threshold_pct=th, max_setup_hold_bars=max_bars)
        
        # max_bars definiuje również jak długo trzymamy trade zanim zamkniemy na czas
        trds = moving_triple_barrier_labels(ents, dfs_1d, tp_mult=tpm, sl_mult=slm, tp_trail_mult=ttpm, max_holding_bars=max_bars)
        
        if len(trds) > 0:
            win_rate = (trds['return_pct'] > 0).mean() * 100
            avg_return = trds['return_pct'].mean()
            results.append({
                'threshold_pct': th,
                'max_holding_bars': max_bars,
                'trades': len(trds),
                'win_rate': win_rate,
                'avg_return': avg_return
            })
            
    df_res = pd.DataFrame(results)
    if not df_res.empty:
        df_res = df_res.sort_values(by='avg_return', ascending=False)
    return df_res

# Odkomentuj aby uruchomić optymalizację
thresholds = [9,10,11,12]
max_holding_bars = [15]

print("Uruchamianie optymalizacji progu...")
opt_df = optimize_threshold(thresholds, max_holding_bars)

display(opt_df.head(10))



Uruchamianie optymalizacji progu...


Optymalizacja progu:   0%|          | 0/4 [00:00<?, ?it/s]

""


In [158]:
# 7. Optymalizacja Parametrów TBM (Take Profit / Stop Loss / Max Hold)
import itertools
from tqdm.notebook import tqdm
import pandas as pd

def optimize_tbm(entries_df, market_data_daily, tp_mults, sl_mults, trail_activations, max_holding_bars_list):
    results = []
    
    # Tworzymy siatkę wszystkich kombinacji
    grid = list(itertools.product(tp_mults, sl_mults, trail_activations, max_holding_bars_list))
    
    for tp, sl, trail, max_bars in tqdm(grid, desc="Optymalizacja TBM"):
        trds = moving_triple_barrier_labels(
            entries_df=entries_df, 
            market_data_daily=market_data_daily, 
            tp_mult=tp, 
            sl_mult=sl, 
            tp_trail_mult=trail, 
            max_holding_bars=max_bars
        )
        
        if len(trds) > 0:
            win_rate = (trds['return_pct'] > 0).mean() * 100
            avg_return = trds['return_pct'].mean()
            avg_hold_bars = trds['hold_bars'].mean()

            results.append({
                'tp_mult': tp,
                'sl_mult': sl,
                'trail_activation': trail,
                'max_holding_bars': max_bars,
                'trades': len(trds),
                'win_rate': win_rate,
                'avg_return': avg_return,
                'avg_hold_bars': avg_hold_bars,
                'return_per_bar': avg_return / avg_hold_bars
            })

    df_res = pd.DataFrame(results)
    if not df_res.empty:
        df_res = df_res.sort_values(by='return_per_bar', ascending=False)
    return df_res


tp_mults = [a/100 for a in range(113, 117, 1)]
sl_mults = [a/100 for a in range(313, 321, 1)]
trail_activations = [a/100 for a in range(1, 3, 1)]
max_holding_bars = [15]

print("Uruchamianie optymalizacji TBM. To może zająć chwilę...")
tbm_opt_results = optimize_tbm(entries_df, dfs_1d, tp_mults, sl_mults, trail_activations, max_holding_bars)

display(tbm_opt_results.head(10))



Uruchamianie optymalizacji TBM. To może zająć chwilę...


Optymalizacja TBM:   0%|          | 0/64 [00:00<?, ?it/s]

,tp_mult,sl_mult,trail_activation,max_holding_bars,trades,win_rate,avg_return,avg_hold_bars,return_per_bar
2,1.13,3.14,0.01,15,107,85.046729,6.695879,7.158879,0.935325
4,1.13,3.15,0.01,15,107,85.046729,6.692163,7.158879,0.934806
6,1.13,3.16,0.01,15,107,85.046729,6.688447,7.158879,0.934287
8,1.13,3.17,0.01,15,107,85.046729,6.684731,7.158879,0.933768
10,1.13,3.18,0.01,15,107,85.046729,6.681015,7.158879,0.933249
18,1.14,3.14,0.01,15,107,85.046729,6.697530,7.177570,0.933119
12,1.13,3.19,0.01,15,107,85.046729,6.677299,7.158879,0.932730
20,1.14,3.15,0.01,15,107,85.046729,6.693814,7.177570,0.932602
14,1.13,3.20,0.01,15,107,85.046729,6.673583,7.158879,0.932211
22,1.14,3.16,0.01,15,107,85.046729,6.690098,7.177570,0.932084
